# 05 - Model Training, Evaluation, and Batch Deployment

This notebook satisfies the Week 4 project steps:

1. Set up a simple benchmark model.
2. Train a first real sentiment model in SageMaker.
3. Evaluate the trained model against the benchmark.
4. Deploy the trained model with SageMaker Batch Transform.

The workflow uses the train, validation, test, and production CSVs created by `03_feature_engineering_feature_store.ipynb` and verified by `04_dataset_splits.ipynb`.


In [ ]:
import json
import tarfile
import tempfile
from pathlib import Path

import boto3
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sagemaker
import seaborn as sns
from sagemaker.inputs import TrainingInput
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.model import SKLearnModel
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

from plot_style import PURPLE_DARK, PURPLE_LIGHT, apply_style

apply_style()
pd.set_option("display.max_colwidth", 120)


## 1. Configure SageMaker and S3 paths

The earlier notebooks store shared variables with `%store`. If a variable is missing, this cell falls back to the known project defaults from the completed Week 3 run.


In [ ]:
%store -r bucket
%store -r role
%store -r region

sagemaker_session = sagemaker.Session()
boto_session = boto3.Session()

region = globals().get("region") or boto_session.region_name
role = globals().get("role") or sagemaker.get_execution_role()
bucket = globals().get("bucket") or f"yelp-sentiment-mlops-{boto3.client('sts').get_caller_identity()['Account']}"

split_prefix = "processed/splits"
model_prefix = "models/week4-sklearn-sentiment"
batch_prefix = "batch/week4-sentiment"
report_dir = Path("../reports")
report_dir.mkdir(parents=True, exist_ok=True)

train_s3 = f"s3://{bucket}/{split_prefix}/train/train.csv"
validation_s3 = f"s3://{bucket}/{split_prefix}/validation/validation.csv"
test_s3 = f"s3://{bucket}/{split_prefix}/test/test.csv"
production_s3 = f"s3://{bucket}/{split_prefix}/production/production.csv"
batch_input_s3 = f"s3://{bucket}/{batch_prefix}/input/production_text.txt"
batch_output_s3 = f"s3://{bucket}/{batch_prefix}/output/"

print("Region:", region)
print("Bucket:", bucket)
print("Role:", role)
print("Train:", train_s3)
print("Validation:", validation_s3)
print("Test:", test_s3)
print("Production:", production_s3)


## 2. Load split data

These are the materialized split files from S3. The production split is reserved for batch scoring to simulate production inference.


In [ ]:
train_df = pd.read_csv(train_s3)
validation_df = pd.read_csv(validation_s3)
test_df = pd.read_csv(test_s3)
production_df = pd.read_csv(production_s3)

for name, df in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
    "production": production_df,
}.items():
    print(f"{name:10s} rows={len(df):,} positive_share={df['sentiment_label'].mean():.3f}")

train_df.head()


## 3. Benchmark model

The benchmark is intentionally simple. It predicts the majority class observed in the training set. This gives us a minimum baseline that the trained model must beat.


In [ ]:
LABELS = [0, 1]

def metric_row(model_name, split_name, y_true, y_pred):
    return {
        "model": model_name,
        "split": split_name,
        "row_count": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

majority_label = int(train_df["sentiment_label"].mode().iloc[0])
benchmark_rows = []
benchmark_confusion = None

for split_name, df in {"validation": validation_df, "test": test_df}.items():
    y_true = df["sentiment_label"].astype(int)
    y_pred = np.full(len(df), majority_label)
    benchmark_rows.append(metric_row("majority_class_benchmark", split_name, y_true, y_pred))
    if split_name == "test":
        benchmark_confusion = confusion_matrix(y_true, y_pred, labels=LABELS)

benchmark_metrics = pd.DataFrame(benchmark_rows)
display(benchmark_metrics)
print("Benchmark majority label:", majority_label)


## 4. Train the first real model in SageMaker

This launches a SageMaker SKLearn training job using `src/train_sklearn.py`. The model is a TF-IDF vectorizer plus Logistic Regression classifier.


In [ ]:
estimator = SKLearn(
    entry_point="train_sklearn.py",
    source_dir="../src",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sagemaker_session,
    output_path=f"s3://{bucket}/{model_prefix}/artifacts",
    hyperparameters={
        "max-features": 50000,
        "ngram-max": 2,
        "c": 1.0,
        "max-iter": 1000,
    },
)

training_inputs = {
    "train": TrainingInput(train_s3, content_type="text/csv"),
    "validation": TrainingInput(validation_s3, content_type="text/csv"),
    "test": TrainingInput(test_s3, content_type="text/csv"),
}

estimator.fit(training_inputs)
model_artifact_s3 = estimator.model_data
print("Model artifact:", model_artifact_s3)


## 5. Compare benchmark and trained model

The training script writes `metrics.json` into the model artifact. This cell downloads the artifact, extracts the metrics, and saves comparison outputs under `reports/`.


In [ ]:
def download_s3_uri(s3_uri, local_path):
    if not s3_uri.startswith("s3://"):
        raise ValueError(f"Expected S3 URI, got {s3_uri}")
    bucket_name, key = s3_uri.replace("s3://", "", 1).split("/", 1)
    boto3.client("s3").download_file(bucket_name, key, str(local_path))

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    artifact_path = tmpdir / "model.tar.gz"
    extract_dir = tmpdir / "model"
    download_s3_uri(model_artifact_s3, artifact_path)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(artifact_path, "r:gz") as tar:
        tar.extractall(extract_dir)
    trained_metrics = json.loads((extract_dir / "metrics.json").read_text())
    trained_model = joblib.load(extract_dir / "model.joblib")

trained_rows = []
for split_name in ["validation", "test"]:
    metrics = trained_metrics[split_name]
    trained_rows.append(
        {
            "model": "tfidf_logistic_regression",
            "split": split_name,
            "row_count": metrics["row_count"],
            "accuracy": metrics["accuracy"],
            "precision_macro": metrics["precision_macro"],
            "recall_macro": metrics["recall_macro"],
            "f1_macro": metrics["f1_macro"],
        }
    )

metrics_df = pd.concat([benchmark_metrics, pd.DataFrame(trained_rows)], ignore_index=True)
metrics_path = report_dir / "benchmark_vs_model_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
display(metrics_df)
print("Wrote", metrics_path)


In [ ]:
y_test = test_df["sentiment_label"].astype(int)
y_test_pred = trained_model.predict(test_df["clean_text"].fillna("").astype(str))
model_confusion = confusion_matrix(y_test, y_test_pred, labels=LABELS)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(
    benchmark_confusion,
    annot=True,
    fmt="d",
    cmap="Purples",
    xticklabels=["negative", "positive"],
    yticklabels=["negative", "positive"],
    ax=axes[0],
)
axes[0].set_title("Benchmark")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(
    model_confusion,
    annot=True,
    fmt="d",
    cmap="Purples",
    xticklabels=["negative", "positive"],
    yticklabels=["negative", "positive"],
    ax=axes[1],
)
axes[1].set_title("TF-IDF Logistic Regression")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
confusion_path = report_dir / "confusion_matrix.png"
plt.savefig(confusion_path, dpi=150)
plt.show()
print("Wrote", confusion_path)


## 6. Batch Transform deployment

The production split is written as one cleaned review per line and scored with SageMaker Batch Transform. This avoids keeping a real-time endpoint running while still demonstrating deployment.


In [ ]:
production_text = production_df["clean_text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True)
local_batch_input = Path("/tmp/production_text.txt")
local_batch_input.write_text("\n".join(production_text.tolist()) + "\n", encoding="utf-8")

bucket_name, key_prefix = batch_input_s3.replace("s3://", "", 1).split("/", 1)
boto3.client("s3").upload_file(str(local_batch_input), bucket_name, key_prefix)
print("Uploaded batch input:", batch_input_s3)

sklearn_model = SKLearnModel(
    model_data=model_artifact_s3,
    role=role,
    entry_point="inference.py",
    source_dir="../src",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sagemaker_session,
)

transformer = sklearn_model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=batch_output_s3,
    accept="application/json",
    assemble_with="Line",
)

transformer.transform(
    data=batch_input_s3,
    content_type="text/plain",
    split_type="Line",
    wait=True,
)

print("Batch Transform output:", transformer.output_path)


## 7. Save Week 4 summary

This summary can be used in the Team Tracker and final deliverables.


In [ ]:
    best_test = metrics_df[(metrics_df["model"] == "tfidf_logistic_regression") & (metrics_df["split"] == "test")].iloc[0]
    benchmark_test = metrics_df[(metrics_df["model"] == "majority_class_benchmark") & (metrics_df["split"] == "test")].iloc[0]

    summary_lines = [
        "# Week 4 Model Evaluation and Deployment Summary",
        "",
        "## Required Steps",
        "",
        "- Simple benchmark model: majority-class benchmark.",
        "- First real model: SageMaker SKLearn TF-IDF + Logistic Regression training job.",
        "- Evaluation: accuracy, macro precision, macro recall, macro F1, and confusion matrix.",
        "- Deployment: SageMaker Batch Transform using the reserved production split.",
        "",
        "## Test Metrics",
        "",
        f"- Benchmark macro F1: {benchmark_test['f1_macro']:.4f}",
        f"- Trained model macro F1: {best_test['f1_macro']:.4f}",
        f"- Trained model accuracy: {best_test['accuracy']:.4f}",
        "",
        "## Artifacts",
        "",
        f"- Model artifact: `{model_artifact_s3}`",
        f"- Batch input: `{batch_input_s3}`",
        f"- Batch output: `{transformer.output_path}`",
        "- Metrics CSV: `reports/benchmark_vs_model_metrics.csv`",
        "- Confusion matrix: `reports/confusion_matrix.png`",
    ]

    summary_path = report_dir / "model_evaluation_summary.md"
    summary_path.write_text("
".join(summary_lines) + "
", encoding="utf-8")
    print("Wrote", summary_path)
    print("
".join(summary_lines))
